In [ ]:
# Install Libraries
!pip install numpy scipy scikit-learn --quiet
# !pip install --upgrade pandas numpy

# Import libraries
import numpy as np
import scipy
import scipy.io #enables uploading of .mat files
import matplotlib.pyplot as plt
# from sklearn.feature_selection import mutual_info_classif #for MI analysis
import math
from scipy import stats as st
from scipy import signal
from scipy import interpolate
from scipy.io import loadmat
print("Libraries loaded successfully!")

Libraries loaded successfully!


This code was adapted from Delia's code which originated from Willet et als github and ganguli labs github. I changed delias code by asking chatGPT:

**"can you help me fix the code so that the axes of the first plot correctly plot original vs warped time with the same range (should be up to 1 second) and that the other plots are graphing the 2nd principal component of the data as opposed to the raw data from the 45th electrode?"**

To apply the Piecewise linear time warping package from Ganguli Lab to replicate the time warping, I (Delia) adopted the codes from Willet et al Github repository for the original paper and used help from ChatGPT. Specifically, the helper function, pre-processing, and normalizing parts were copied here directly from the authors. To time warp the dataset, I've given the following prompts to ChatGPT:

**"Can you give me a code example using affinewarp to time-warp the neural trials from the singleLetters.mat file?"**

**"This is the time warping codes the original paper was using to their singleLetters.mat files across all the sessions, how can I apply their codes but using the affinewarp package to conduct my own time warping?"**

The GitHub repository for Piecewise linear time warping from [Ganguli Lab](https://github.com/ahwillia/affinewarp/tree/master) and from [Willet et al](https://github.com/fwillett/handwritingBCI/blob/93d6cc65491785d2743c74948b75beb0e2eaaeb1/Step1_timeWarp.ipynb) were also referenced.

In [ ]:
# to load files from google drive directly
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Time-warping the singleLetters data

# Install and load the updated time-warping package ("Piecewise Linear Time Warping")
!pip install git+https://github.com/ahwillia/affinewarp.git

import affinewarp as aw
import scipy.io
from scipy.ndimage import gaussian_filter1d
from affinewarp.piecewisewarp import PiecewiseWarping # Piecewise warping is the closest warping function to the TWPCA done in the paper
import os

dat = scipy.io.loadmat('/content/drive/MyDrive/Emory_Year_2/COMP NEURO/t5.2019.05.08_singleLetters.mat')

  Cloning https://github.com/ahwillia/affinewarp.git to /tmp/pip-req-build-hxxf81kw
  Running command git clone --filter=blob:none --quiet https://github.com/ahwillia/affinewarp.git /tmp/pip-req-build-hxxf81kw
  Resolved https://github.com/ahwillia/affinewarp.git to commit 23f9e643d2e74ad930ef283311c2c14c585eb6b9
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for affinewarp: filename=affinewarp-0.2.0-py3-none-any.whl size=37756 sha256=b8df1b840fac84d8f96a8a57ace29b9b3f8f11311bd207104e571915f726652f
  Stored in directory: /tmp/pip-ephem-wheel-cache-l5kx6eof/wheels/ef/f8/7a/4c13e469527a873053587957a36091f26bd60219a904485045
Successfully built affinewarp


In [ ]:
# Helper Function

def getHandwritingCharacterDefinitions():
  """
  Returns a dictionary with entries that define the names of each character, its length, and where the pen tip begins.

  Returns:
      charDef(dict)
  """

  charDef = {}

  # Define the list of all 31 characters and their names
  charDef['charList'] = ['a','b','c','d','e','f','g','h','i','j','k','l','m','n','o','p','q','r','s','t','u','v','w','x','y','z',
                'greaterThan','comma','apostrophe','tilde','questionMark']
  charDef['charListAbbr'] = ['a','b','c','d','e','f','g','h','i','j','k','l','m','n','o','p','q','r','s','t','u','v','w','x','y','z',
                '>',',',"'",'~','?']

  # Define the length of each character (in # of 10 ms bins) to use for each template.
  # These were hand-defined based on visual inspection of the reconstructed pen trajectories.
  charDef['charLen'] = np.array([99, 91, 70, 104, 98, 125, 110, 104, 79, 92, 127, 68, 132, 90,
                        84, 113, 104, 74, 86, 110, 86, 83, 110, 103, 115, 100, 82, 77, 116, 71, 110]).astype(np.int32)

  # For each character, this defines the starting location of the pen tip (0 = bottom of the line, 1 = top)
  charDef['penStart'] = [0.25, 1, 0.5, 0.5, 0.25, 1.0, 0.25, 1.0, 0.5, 0.5, 1, 1, 0.5, 0.5, 0.25, 0.5, 0.25, 0.5, 0.5, 1,
           0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.25, 1, 0.5, 1]

  # Dictionary to convert string representation to character index
  charDef['strToCharIdx'] = {}
  for x in range(len(charDef['charListAbbr'])):
    charDef['strToCharIdx'][charDef['charListAbbr'][x]] = x

  return charDef

In [ ]:
# Time-warping the single letters data cont'd: VERY SLOW (~23 mins on my end)

# defines the list of all 31 characters and what to call then
charDef = getHandwritingCharacterDefinitions()

# Pre-processing and Normalizing
# Because baseline firing rates drift over time, we normalize each electrode's firing rate by subtracting its mean firing rate within each block of data (re-centering it).
# We also divide by each electrode's standard deviation to normalize the units.

for char in charDef['charList']:
    neuralCube = dat['neuralActivityCube_' + char].astype(np.float64)

    # get the trials that belong to this character
    trlIdx = []
    for t in range(dat['characterCues'].shape[0]):
      if dat['characterCues'][t,0] == char:
        trlIdx.append(t)
    # get the block that each trial
    blockIdx = dat['blockNumsTimeSeries'][dat['goPeriodOnsetTimeBin'][trlIdx]]
    blockIdx = np.squeeze(blockIdx)

    # subtract block-specific means from each trial
    for b in range(dat['blockList'].shape[0]):
      trialsFromThisBlock = np.squeeze(blockIdx == dat['blockList'][b])
      neuralCube[trialsFromThisBlock, :, :] -= dat['meansPerBlock'][np.newaxis, b, :]

    # divide by standard deviation to normalize the units
    neuralCube = neuralCube / dat['stdAcrossAllData'][np.newaxis, :, :]

    # save the normalized neural cube in the same dataset
    dat['normalized_neuralActivityCube_' + char] = neuralCube

# Warp each character

alignedDat = {}

for char in charDef['charList']:
    print('Warping character: ' + char)

    # smooths the binned spike counts before time-warping to denoise them
    smoothed_spikes = scipy.ndimage.filters.gaussian_filter1d(dat['normalized_neuralActivityCube_' + char], 3.0, axis = 1)

    # fit the piecewise-affine time warping model
    model = PiecewiseWarping(
    n_knots=5,
    warp_reg_scale=0.001,       # encourages identity warping
    smoothness_reg_scale=1.0, # makes template smoother
    l2_reg_scale=1e-7          # minimal template L2 norm
)
    model.fit(smoothed_spikes)

    # use the model object to align data
    estimated_aligned_data = model.transform(dat['normalized_neuralActivityCube_' + char])
    smoothed_agligned_data = scipy.ndimage.filters.gaussian_filter1d(estimated_aligned_data, 3.0, axis = 1)
    # === (1) Plot the warping functions (like the authors do) ===
    plt.figure(figsize=(14, 4))
    plt.subplot(1, 3, 1)
    plt.plot(model.x_knots.T, model.y_knots.T, alpha=1)
    plt.axis('square')
    plt.xlabel('Original Time')
    plt.ylabel('Aligned Time')
    plt.title(f'Warping Functions - {char}')
    plt.xlim(0, 1)
    plt.ylim(0, 1)

    # === (2) PCA projection plot for PC2 across trials ===
    from sklearn.decomposition import PCA

    # Use unwarped smoothed data to compute PCA basis
    trial_avg = np.mean(smoothed_spikes, axis=0)
    pca = PCA(n_components=3)
    pca.fit(trial_avg)  # [T, N]
    neuron_factors = pca.components_.T  # [N, PCs]

    # Unwarped projection
    plt.subplot(1, 3, 2)
    for t in range(smoothed_spikes.shape[0]):
        proj = smoothed_spikes[t] @ neuron_factors
        plt.plot(proj[:, 1], alpha=0.5)
    plt.title(f"Unwarped PC2 - {char}")
    plt.xlabel("Time Bins")
    plt.ylabel("PC2")
    # Use unwarped smoothed data to compute PCA basis
    trial_avg = np.mean(smoothed_agligned_data, axis=0)
    pca = PCA(n_components=3)
    pca.fit(trial_avg)  # [T, N]
    neuron_factors = pca.components_.T  # [N, PCs]

    # Warped projection
    plt.subplot(1, 3, 3)
    for t in range(smoothed_agligned_data.shape[0]):
        proj = smoothed_agligned_data[t] @ neuron_factors
        plt.plot(proj[:, 1], alpha=0.5)
    plt.title(f"Warped PC2 - {char}")
    plt.xlabel("Time Bins")
    plt.ylabel("PC2")

    plt.tight_layout()
    plt.show()
    # store aligned data and time-warping functions
    alignedDat[char] = estimated_aligned_data
    alignedDat[char + '_xknots'] = model.x_knots.T
    alignedDat[char + '_yknots'] = model.y_knots.T



# Save time-warped data
scipy.io.savemat('/content/drive/MyDrive/Emory_Year_2/COMP NEURO/t5.2019.05.08_warpedCubes_4_30.mat', alignedDat)

Output hidden; open in https://colab.research.google.com to view.